# ptq-bench on Google Colab

Measure how much a language model loses when its weights are squeezed to fewer bits, on a
free GPU, without installing anything on your own computer.

**Before you start:** in the menu above, choose *Runtime → Change runtime type* and pick a
GPU (a T4 is enough). Then run the cells in order with the ▶ buttons. The whole notebook
takes about five minutes. Colab wipes everything when the session ends; the numbers you
see are the same ones the project measures on real hardware, and are compared with the
published papers.


## 1. Get the project and set it up (about two minutes)

In [ ]:
%cd /content
!git clone -q https://github.com/Dercen/LLM-Precision.git
%cd /content/LLM-Precision
!./install.sh 2>&1 | tail -25

## 2. Is this machine ready?

Green lines are fine. A yellow line is a note, a red line needs fixing (the fix is printed next to it).

In [ ]:
!./ptq doctor

## 3. The two-minute demo

Two quick previews of opt-125m on Wikipedia text: first the unmodified model, then the same
model at 4 bits with plain rounding. The second number is read against the first, and the
sentence at the end says what it means.

In [ ]:
!.venv/bin/python - <<'EOF'
from ptqbench import explain as E, wizard as W
for a in W.demo_answers():
    print("Plan:", W._summary(a), "| estimated time:", W.estimate(a))
    rows = W.execute_answers(a, log=lambda m: print("  " + E.plain(m)))
    for r in rows:
        baseline = E.baseline_ppl(r["model"], r["dataset"], allow_partial=True)
        print("  perplexity:", round(r["ppl"], 4))
        print("  What this means:", E.verdict(r, baseline))
    print()
EOF

## 4. A real measurement (optional, about three minutes on a T4)

Every passage of the test text instead of 20, with GPTQ, the method that corrects its own
rounding errors. This number is directly comparable with the GPTQ paper's.

In [ ]:
!.venv/bin/python - <<'EOF'
from ptqbench import explain as E, wizard as W
a = W.Answers(model_key="opt-125m", datasets=["wikitext2"], algo="gptq", bits=4, group_size=128, calib_dataset="wikitext2")
print("Plan:", W._summary(a), "| estimated time:", W.estimate(a))
for r in W.execute_answers(a, log=lambda m: print("  " + E.plain(m))):
    paper, source = W.paper_number(r)
    print("  perplexity:", round(r["ppl"], 4), "| paper:", paper, source)
    print("  What this means:", E.verdict(r, E.baseline_ppl(r["model"], r["dataset"])))
EOF

## 5. The results pages, rebuilt with your runs in them

In [ ]:
!./ptq aggregate && ./ptq plot --models opt-125m
from IPython.display import Image, Markdown, display
display(Markdown(open("results/by-model/opt-125m.md").read().split("## Charts")[0]))
display(Image("results/plots/opt-125m_perplexity.png"))

## Try your own

Change `model_key` (any name from `configs/models/`: opt-350m, smollm2-135m, ...), `algo`
(`rtn`, `gptq`, `awq_lite`, `hqq`), `bits` (8, 4, 3, 2) or `group_size` (128, 64, -1 for
per row) in step 4 and run it again. Bigger models need more memory than a free Colab GPU
has; opt-125m to opt-1.3b fit. To run this on your own machine, the
[README](https://github.com/Dercen/LLM-Precision#start-here) has a one-command setup.
